# Stage 3 Helmet Training (Google Colab)

Use a Colab GPU runtime. Upload the prepared dataset as `MyDrive/traffic-ai/datasets/helmet.zip` (the zip must contain `images/`, `labels/`, and `data.yaml`, either at its root or inside a single `helmet/` folder). The source dataset and labels remain unchanged. This notebook extracts the zip to Colab's local disk for faster training, uses `yolo26n.pt`, 40 epochs, 640 image size, batch 16, workers 2, and patience 10, then saves the best model and run outputs back to Drive.


In [ ]:
!pip -q install ultralytics
from google.colab import drive
drive.mount('/content/drive')
import shutil, yaml, zipfile
from pathlib import Path
drive_project = Path('/content/drive/MyDrive/traffic-ai')
drive_zip = drive_project / 'datasets' / 'helmet.zip'
drive_data = drive_project / 'datasets' / 'helmet'
local_data = Path('/content/helmet')
if local_data.exists(): shutil.rmtree(local_data)
if drive_zip.exists():
    with zipfile.ZipFile(drive_zip) as zf: zf.extractall('/content/helmet_unpack')
    candidates = list(Path('/content/helmet_unpack').rglob('data.yaml'))
    if not candidates: raise FileNotFoundError('helmet.zip must contain data.yaml')
    shutil.copytree(candidates[0].parent, local_data)
elif (drive_data / 'data.yaml').exists():
    shutil.copytree(drive_data, local_data)
else:
    raise FileNotFoundError(f'Upload helmet.zip or the helmet folder under {drive_project}/datasets')
cfg = yaml.safe_load((local_data / 'data.yaml').read_text())
cfg.update(path=str(local_data), train='images/train', val='images/val', test='images/test')
(local_data / 'data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print('GPU:', __import__('torch').cuda.get_device_name(0))
print('Dataset:', local_data)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n.pt')
train_results = model.train(
    data=str(local_data / 'data.yaml'), epochs=40, imgsz=640, batch=16, workers=2,
    patience=10, project='/content/runs', name='helmet_v1', device=0
)

In [ ]:
best = Path('/content/runs/helmet_v1/weights/best.pt')
assert best.exists(), 'Training did not produce best.pt'
out_dir = Path('/content/drive/MyDrive/traffic-ai/models/helmet')
out_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(best, out_dir / 'helmet_v1.pt')
shutil.copytree('/content/runs/helmet_v1', '/content/drive/MyDrive/traffic-ai/runs/helmet_v1', dirs_exist_ok=True)
print('Saved:', out_dir / 'helmet_v1.pt')

Download `helmet_v1.pt` from Drive and place it at `models/helmet/helmet_v1.pt` in the project, then run `python scripts/s3_evaluate_helmet.py`. Review the test results and validation previews before treating the model as useful.